# Logical Planning Laboratory – Logic + Search = Planning

**Contents:** Task 0 (problem) · Task 1 (hand plan) · Task 2 (LLM planner) · Task 3 (tests) · Task 4 (logic and search) · Task 5 (can the LLM verify itself?) · Reflection questions · Optional Prolog extension (Tasks 6–8)

## Task 0 – Understand the Planning Problem

Planning problem **(I, A, G)**.

**(a) Initial state**  `I = {At(Robot,A), At(Package,A)}`

**(b) Goal**  `G = {At(Package,C)}`

**(c–d) Actions, preconditions and effects**

| Action | Preconditions | Effects (negative / positive) |
|---|---|---|
| `Move(A,B)` | At(Robot,A) | ¬At(Robot,A), At(Robot,B) |
| `Move(B,A)` | At(Robot,B) | ¬At(Robot,B), At(Robot,A) |
| `Move(B,C)` | At(Robot,B) | ¬At(Robot,B), At(Robot,C) |
| `Move(C,B)` | At(Robot,C) | ¬At(Robot,C), At(Robot,B) |
| `PickUp(Package,L)` for L ∈ {A,B,C} | At(Robot,L), At(Package,L) | ¬At(Package,L), Holding(Package) |
| `Drop(Package,L)` for L ∈ {A,B,C} | At(Robot,L), Holding(Package) | ¬Holding(Package), At(Package,L) |

There are only Move actions between *connected* locations (A–B and B–C). There is no A–C connection.

*Assumption:* the sheet gives `PickUp(Package,A)` and `Drop(Package,C)` as examples; I generalise them to every location L so the planner can also handle intermediate drops and pickups.

**Which actions are initially applicable?** A state is a set of true propositions; an action is applicable if `S ⊨ Preconditions(a)`.

- **`PickUp(Package,A)` – applicable.** Both preconditions, `At(Robot,A)` and `At(Package,A)`, are in `I`.
- **`Drop(Package,C)` – not applicable.** It needs `At(Robot,C)` and `Holding(Package)`, and neither is true in `I` (the robot is at A and holds nothing).

Being in the list of available actions is not enough: every precondition must be satisfied in the *current* state.

## Task 1 – Construct a Plan by Hand

| State | Facts | Action that produced it |
|---|---|---|
| S0 | At(Robot,A), At(Package,A) | – |
| S1 | At(Robot,A), Holding(Package) | `PickUp(Package,A)` |
| S2 | At(Robot,B), Holding(Package) | `Move(A,B)` |
| S3 | At(Robot,C), Holding(Package) | `Move(B,C)` |
| S4 | At(Robot,C), At(Package,C) | `Drop(Package,C)` |

`S4 ⊨ G`, so the plan is **PickUp(Package,A), Move(A,B), Move(B,C), Drop(Package,C)** (4 actions).

**Warning about the sheet's example.** The sheet suggests reasoning about `Move(A,B), PickUp(Package,B), Move(B,C), Drop(Package,C)`. That sequence is **not valid**: after `Move(A,B)` the robot is at B, but the package is still at A, so `PickUp(Package,B)` needs `At(Package,B)`, which is false. The robot must pick the package up *before* leaving A. The code below checks this automatically.

## Task 2 – Ask an LLM to Implement the Planner

### Prompt used
> I want to implement a simple planning agent in Python. Represent a state as a set of logical propositions. Each action should contain: a name; positive preconditions; negative preconditions; positive effects; negative effects. An action is applicable if all of its preconditions are satisfied by the current state. When an action is applied: (1) remove its negative effects from the state; (2) add its positive effects to the state. Use breadth-first search to find a sequence of actions that achieves a specified goal. The program should also: detect when no plan exists; print the resulting sequence of actions; print the states reached after each action. Explain the implementation and identify any assumptions you make.

*Follow-ups:* "Write a separate function that replays a plan and checks every precondition", "add a flag to switch off precondition checking (for a demonstration)".

### Assumptions
- A state is a `frozenset` of true propositions (strings such as `"At(Robot,A)"`). **Closed-world assumption:** anything not in the set is false.
- Effects are applied as *delete first, then add*.
- BFS finds a plan with the fewest actions. The goal is satisfied when `goal ⊆ state`.
- One robot, one package; all actions are deterministic.

### Code
**Action representation**

In [1]:
"""STRIPS-style planning agent (BFS) for the warehouse robot - Logical Planning Lab."""
from collections import deque
from dataclasses import dataclass


@dataclass(frozen=True)
class Action:
    """A planning operator: name, positive/negative preconditions, positive/negative effects."""
    name: str
    pos_pre: frozenset = frozenset()
    neg_pre: frozenset = frozenset()
    pos_eff: frozenset = frozenset()
    neg_eff: frozenset = frozenset()

    def __str__(self):
        return self.name


def fs(*props):
    return frozenset(props)

**The warehouse domain** – locations, connections and action constructors

In [2]:
LOCATIONS = ["A", "B", "C"]
CONNECTED = [("A", "B"), ("B", "A"), ("B", "C"), ("C", "B")]


def move(x, y):
    return Action(f"Move({x},{y})",
                  pos_pre=fs(f"At(Robot,{x})"),
                  pos_eff=fs(f"At(Robot,{y})"),
                  neg_eff=fs(f"At(Robot,{x})"))


def pickup(l):
    return Action(f"PickUp(Package,{l})",
                  pos_pre=fs(f"At(Robot,{l})", f"At(Package,{l})"),
                  pos_eff=fs("Holding(Package)"),
                  neg_eff=fs(f"At(Package,{l})"))


def drop(l):
    return Action(f"Drop(Package,{l})",
                  pos_pre=fs(f"At(Robot,{l})", "Holding(Package)"),
                  pos_eff=fs(f"At(Package,{l})"),
                  neg_eff=fs("Holding(Package)"))


def warehouse_actions(connected=CONNECTED, with_pickup=True, with_drop=True):
    acts = [move(x, y) for x, y in connected]
    if with_pickup:
        acts += [pickup(l) for l in LOCATIONS]
    if with_drop:
        acts += [drop(l) for l in LOCATIONS]
    return acts


INITIAL = fs("At(Robot,A)", "At(Package,A)")
GOAL = fs("At(Package,C)")

**Logic** – applicability `S ⊨ Preconditions(a)` and applying effects `S' = Apply(S,a)`

In [3]:
def is_applicable(state, action):
    """S |= Preconditions(a): all positive preconditions hold, no negative precondition holds."""
    return action.pos_pre <= state and action.neg_pre.isdisjoint(state)


def apply_action(state, action):
    """S' = Apply(S, a): remove negative effects first, then add positive effects."""
    return (state - action.neg_eff) | action.pos_eff

**Search** – breadth-first search over sets of propositions

In [4]:
def bfs_plan(initial, goal, actions, check_preconditions=True):
    """Return dict(found, plan, states, expanded). Shortest plan (fewest actions) if one exists."""
    initial, goal = frozenset(initial), frozenset(goal)
    frontier = deque([initial])
    parent = {initial: None}                      # state -> (previous state, action); doubles as 'visited'
    expanded = 0
    while frontier:
        state = frontier.popleft()
        expanded += 1
        if goal <= state:                         # goal test: S |= G
            plan, states, node = [], [state], state
            while parent[node] is not None:
                node, act = parent[node]
                plan.append(act)
                states.append(node)
            plan.reverse(); states.reverse()
            return {"found": True, "plan": plan, "states": states, "expanded": expanded}
        for act in actions:
            if check_preconditions and not is_applicable(state, act):
                continue                          # logic prunes impossible actions
            nxt = apply_action(state, act)
            if nxt not in parent:                 # never revisit a state
                parent[nxt] = (state, act)
                frontier.append(nxt)
    return {"found": False, "plan": None, "states": None, "expanded": expanded}


def fmt(state):
    return ", ".join(sorted(state)) if state else "(empty)"


def show(result, goal=None):
    if not result["found"]:
        print(f"No plan found  (states expanded: {result['expanded']})")
        return
    print(f"Plan found with {len(result['plan'])} actions (states expanded: {result['expanded']})")
    print(f"  S0: {fmt(result['states'][0])}")
    for i, (a, s) in enumerate(zip(result["plan"], result["states"][1:]), 1):
        print(f"  a{i}: {a}\n  S{i}: {fmt(s)}")

**Independent plan checker and helpers** – replays a plan and checks every precondition explicitly, separately from the search code

In [5]:
def verify_plan(initial, plan, goal, verbose=True):
    """Replay a plan step by step, checking every precondition explicitly.
    Written separately from bfs_plan so it can act as an independent check on it."""
    state = set(initial)
    for i, act in enumerate(plan, 1):
        missing = [p for p in sorted(act.pos_pre) if p not in state]
        blocked = [p for p in sorted(act.neg_pre) if p in state]
        if missing or blocked:
            if verbose:
                print(f"  step {i}: {act} INVALID - missing {missing or '-'}, forbidden-but-true {blocked or '-'}")
            return False
        state.difference_update(act.neg_eff)
        state.update(act.pos_eff)
        if verbose:
            print(f"  step {i}: {act} ok -> {{{fmt(state)}}}")
    ok = set(goal) <= state
    if verbose:
        print(f"  goal {{{fmt(goal)}}} satisfied: {ok}")
    return ok


def reachable_states(initial, actions):
    initial = frozenset(initial)
    seen, queue = {initial}, deque([initial])
    while queue:
        s = queue.popleft()
        for a in actions:
            if is_applicable(s, a):
                n = apply_action(s, a)
                if n not in seen:
                    seen.add(n); queue.append(n)
    return seen

### Run the planner on the warehouse problem

In [6]:
actions = warehouse_actions()
print("Applicable in I:", [str(a) for a in actions if is_applicable(INITIAL, a)])
print("Is Drop(Package,C) applicable in I?", is_applicable(INITIAL, drop("C")))
print()
result = bfs_plan(INITIAL, GOAL, actions)
show(result)

Applicable in I: ['Move(A,B)', 'PickUp(Package,A)']
Is Drop(Package,C) applicable in I? False

Plan found with 4 actions (states expanded: 8)
  S0: At(Package,A), At(Robot,A)
  a1: PickUp(Package,A)
  S1: At(Robot,A), Holding(Package)
  a2: Move(A,B)
  S2: At(Robot,B), Holding(Package)
  a3: Move(B,C)
  S3: At(Robot,C), Holding(Package)
  a4: Drop(Package,C)
  S4: At(Package,C), At(Robot,C)


### "Think About It" – where do the specification's ideas appear in the code?

| Idea | Where in the program |
|---|---|
| **Preconditions → when is an action applicable?** | `is_applicable`: `action.pos_pre <= state and action.neg_pre.isdisjoint(state)`, called inside `bfs_plan` before any successor is generated. |
| **Effects → how does the state change?** | `apply_action`: `(state - action.neg_eff) \| action.pos_eff`. |
| **Goal → when does planning terminate?** | `if goal <= state:` in `bfs_plan`. It stops when the popped state contains all goal propositions, or when the frontier is empty (failure). |
| **BFS → how are alternatives explored?** | The FIFO queue `frontier` (a `deque`): all 1-action plans are tried, then all 2-action plans, and so on. `parent` records the visited states and how each was reached. |

## Task 3 – Test the Generated Planner
Each test uses `assert`, and the returned plan is always checked with the independent `verify_plan`.

### Test A – Solvable problem (original warehouse)

In [ ]:
resA = bfs_plan(INITIAL, GOAL, actions)
show(resA)
print("\nIndependent verification of every step:")
assert verify_plan(INITIAL, resA["plan"], GOAL)
assert [str(a) for a in resA["plan"]] == ["PickUp(Package,A)", "Move(A,B)", "Move(B,C)", "Drop(Package,C)"]
print("Test A passed (matches my hand-made plan from Task 1).")

### Test B – Impossible problems (planner must say *No plan found*)

In [ ]:
# B1: remove the PickUp action
resB1 = bfs_plan(INITIAL, GOAL, warehouse_actions(with_pickup=False))
show(resB1)
assert not resB1["found"]

# B2: keep PickUp but remove the only way to reach C (Move(B,C))
no_bc = [c for c in CONNECTED if c != ("B", "C")]
resB2 = bfs_plan(INITIAL, GOAL, warehouse_actions(connected=no_bc))
show(resB2)
assert not resB2["found"]
print("Test B passed: no invented actions, and it terminated.")

### Test C – Irrelevant actions: robot reaching C is not the package reaching C

In [ ]:
# Goal 1: only the ROBOT must reach C
resC1 = bfs_plan(INITIAL, fs("At(Robot,C)"), actions)
show(resC1)
final = resC1["states"][-1]
print("Final state satisfies At(Package,C)?", "At(Package,C)" in final)
assert "At(Package,C)" not in final               # robot moved, package stayed at A
assert not verify_plan(INITIAL, resC1["plan"], GOAL, verbose=False)   # that plan does NOT achieve G

# Move(A,B) is applicable although the package stays at A - it is legal but irrelevant
after_move = apply_action(INITIAL, move("A", "B"))
print("\nAfter Move(A,B):", fmt(after_move))
assert "At(Package,A)" in after_move

# Goal 2: robot AND package at C -> the plan must include the pickup/drop
resC2 = bfs_plan(INITIAL, fs("At(Robot,C)", "At(Package,C)"), actions)
assert len(resC2["plan"]) == 4
print("\nTest C passed: the planner distinguishes At(Robot,C) from At(Package,C).")

### Extra tests
1. **Preconditions matter.** Deliberately switch precondition checking *off* and see what goes wrong.
2. **The sheet's example sequence** from Task 1 must be rejected.
3. **Negative preconditions** work (the specification requires them, though the warehouse does not use them).
4. **State-space size.**

In [ ]:
# Extra 1: planner WITHOUT precondition checking (a deliberately buggy planner)
bad = bfs_plan(INITIAL, GOAL, actions, check_preconditions=False)
show(bad)
print("Independent check:")
assert not verify_plan(INITIAL, bad["plan"], GOAL)
print("  -> the buggy planner 'solved' it in 1 step, by dropping a package it was not holding, at a place it was not.")
print("  Its final state has the package at A AND C:", fmt(bad["states"][-1]))

# Extra 2: the example sequence from the lab sheet
print("\nSheet's example sequence:")
sheet_plan = [move("A", "B"), pickup("B"), move("B", "C"), drop("C")]
assert not verify_plan(INITIAL, sheet_plan, GOAL)

# Extra 3: negative preconditions
unlock = Action("Unlock", pos_eff=fs("Unlocked"), neg_eff=fs("Locked"))
open_door = Action("OpenDoor", pos_pre=fs("Closed"), neg_pre=fs("Locked"), pos_eff=fs("Open"), neg_eff=fs("Closed"))
s0 = fs("Closed", "Locked")
assert not is_applicable(s0, open_door)                                   # blocked by negative precondition
r = bfs_plan(s0, fs("Open"), [open_door, unlock])
assert r["found"] and [str(a) for a in r["plan"]] == ["Unlock", "OpenDoor"]
print("\nNegative-precondition test passed:", [str(a) for a in r["plan"]])

# Extra 4: how big is the state space?
print("\nReachable states in the warehouse problem:", len(reachable_states(INITIAL, actions)),
      "(3 robot locations x 4 package situations: at A, B, C or held)")

### Record of results

| Test | Initial state | Goal | Plan found? | Plan | Valid? |
|---|---|---|---|---|---|
| **A** solvable | {At(Robot,A), At(Package,A)} | {At(Package,C)} | Yes (8 states expanded) | PickUp(Package,A), Move(A,B), Move(B,C), Drop(Package,C) | Yes, verified step by step |
| **B1** no PickUp | same | same | **No** (3 states expanded) | – | – (correctly reports failure) |
| **B2** no Move(B,C) | same | same | **No** (6 states expanded) | – | – (correctly reports failure) |
| **C1** robot only | same | {At(Robot,C)} | Yes (4 states expanded) | Move(A,B), Move(B,C) | Valid for that goal; **does not achieve** At(Package,C) |
| **C2** both | same | {At(Robot,C), At(Package,C)} | Yes | 4 actions (pickup/drop included) | Yes |
| **Extra 1** no precondition check | same | {At(Package,C)} | "Yes" (wrong!) | Drop(Package,C) | **Invalid**: Robot not at C, not holding |
| **Extra 2** sheet's example | same | {At(Package,C)} | – (checked only) | Move(A,B), PickUp(Package,B), Move(B,C), Drop(Package,C) | **Invalid** at step 2 |

## Task 4 – Logic and Search

**Completing the description**

```
Current state
      ↓
Check action preconditions           ← LOGIC: does S ⊨ Preconditions(a)?
      ↓
If yes, the action is APPLICABLE; if not, discard it (prune)   ← the "?" box
      ↓
Generate successor state             ← LOGIC: S' = (S − negative effects) ∪ positive effects
      ↓
Search over alternatives             ← SEARCH: put S' on the BFS queue unless already visited
      ↓
Goal?                                ← LOGIC: does S' ⊨ G?  If yes stop; if not, take the next state from the queue
```

**Explanation in my own words.** Logic and search do different jobs. *Logic* defines what is *possible*: for a state, it tells which actions are applicable (preconditions hold in the state) and what state each action leads to (effects), and it decides whether a state satisfies the goal. Left alone, logic only answers questions about one state and one action; it does not decide which action to take. *Search* decides what to *try*: it keeps a queue of states, systematically expands them in breadth-first order, avoids revisiting states, and records how each was reached so a plan can be read back. Search without logic would consider impossible moves (like dropping a package you don't hold); logic without search would know what is allowed but not have a strategy for finding the sequence. Together, **Logic + Search = Planning**.

*Logic determines what is possible; search determines what to try.* This is the same as the search module: the state space is the graph, applicability plus effects define its edges, and BFS is the same BFS. The difference is that the graph is not drawn as a grid; it is *generated* from logical action descriptions.

## Task 5 (Optional) – Can the LLM Verify Its Own Plan?

**The LLM's explanation** (as requested: for each action, the preconditions and why they hold):

> 1. `PickUp(Package,A)`. Needs At(Robot,A) and At(Package,A): both in S0 ✔. Result S1 = {At(Robot,A), Holding(Package)}.
> 2. `Move(A,B)`. Needs At(Robot,A): in S1 ✔. Result S2 = {At(Robot,B), Holding(Package)}.
> 3. `Move(B,C)`. Needs At(Robot,B): in S2 ✔. Result S3 = {At(Robot,C), Holding(Package)}.
> 4. `Drop(Package,C)`. Needs At(Robot,C) and Holding(Package): both in S3 ✔. Result S4 = {At(Robot,C), At(Package,C)}, which contains the goal.

Now compare it with what the program actually computes.

In [ ]:
claimed_states = [   # the states asserted in the explanation above, typed in by hand
    {"At(Robot,A)", "At(Package,A)"},
    {"At(Robot,A)", "Holding(Package)"},
    {"At(Robot,B)", "Holding(Package)"},
    {"At(Robot,C)", "Holding(Package)"},
    {"At(Robot,C)", "At(Package,C)"},
]
computed_states = [set(s) for s in resA["states"]]

for i, (claimed, computed) in enumerate(zip(claimed_states, computed_states)):
    print(f"S{i}: explanation == executed transition ? {claimed == computed}")
assert claimed_states == computed_states

**Which should I trust more: (a) the LLM's explanation or (b) the independently executed state transitions?**

**(b).** In this case the two agree, but that agreement is a *fact about this run*, not a guarantee. The LLM's explanation is generated text: it is produced because it is plausible, and it can contain a wrong claim written with the same confidence as a right one. The sheet's own example plan (`PickUp(Package,B)` after `Move(A,B)`) shows how a sequence that reads reasonably fails when the preconditions are checked. The executed transitions are computed by deterministic code that checks each precondition in a state that was actually reached, and can be re-run and tested by anyone. Also, remember that the checker only verifies the plan *against the action model I wrote*. If that model is wrong (say, I forgot a precondition), both pass; the specification itself still needs human review.

*A generated explanation is not the same as an independent verification.*

## Reflection Questions

**1. Why specify preconditions and effects before asking the LLM to write the planner?**
They *are* the specification. If the LLM has to guess when an action is legal or what it changes, it will fill the gaps with plausible-looking assumptions. Writing them first defines what "correct" means, so the generated code can be tested against something concrete (as in Tasks 3 and 5).

**2. Example error if the planner skipped precondition checks?**
The buggy planner in the extra test "solved" the problem with a single action, `Drop(Package,C)`, though the robot was at A and holding nothing. The resulting state also had the package in two places at once (A and C).

**3. Why is a plan that "looks reasonable" not necessarily valid?**
Validity is a property of every step *in the state in which it is executed*, not of how sensible the list of action names reads. The sheet's example `Move(A,B), PickUp(Package,B), ...` reads naturally but fails, because the package is still at A.

**4. What did the LLM contribute?**
The initial BFS/STRIPS-style implementation and structure (`Action` class, frozenset states, `parent` dictionary, BFS loop), the action constructors, the explanations and (in this report) the tests, which I ran and checked.

**5. What did I have to verify independently?**
That the plan is valid step by step (`verify_plan`), that impossible problems really give "No plan found", that reaching C with the robot is not the same as the package reaching C, that precondition checking is what stops bogus plans, and that the action definitions themselves match the problem sheet.

**6. Where is logical reasoning used in this lab?**
In `is_applicable` (`S ⊨ Preconditions(a)`, including negative preconditions), in `apply_action` (computing the successor state from the effects), and in the goal test `S ⊨ G`. Also in the checker `verify_plan`, and in Prolog (Tasks 6–8) where rules are used to infer whether a move is supported.

**7. How is planning related to the search algorithms of the previous module?**
Planning *is* search: states are nodes, applicable actions are edges, each action costs 1, the goal test is `S ⊨ G`, and BFS is the same algorithm as in the earlier lab. The difference is that the state space is implicit and generated by logical action descriptions. A* could be used too, with a heuristic such as "number of unsatisfied goal propositions" (an admissible one for this domain is not trivial to design, which is one reason planners use more careful heuristics).

## Task 6 – Prolog as a plan verifier

| Query | Expected result |
|---|---|
| `?- can_move(a,b).` | `true.` |
| `?- can_move(a,c).` | `false.` |

**(a) Why `true` for `can_move(a,b)`?** Prolog looks for a rule or fact matching `can_move(a,b)`. The rule `can_move(X,Y) :- connected(X,Y)` matches with X=a, Y=b, giving the subgoal `connected(a,b)`, which is a stored fact. So the query is derivable.

**(b) Why is `can_move(a,c)` not established?** The rule would need `connected(a,c)`, and there is no such fact, nor any rule that could derive it (there is no rule for going "via b"). Prolog therefore answers `false`. This means "cannot be proven from this knowledge base" (negation as failure under the closed-world assumption); it is not a proof that a→c is impossible in the real world. The robot can reach c *by a sequence*: `a→b→c`, which the KB records as two separate one-step moves.

**(c) Relationship between the rule and the implication.** `can_move(X,Y) :- connected(X,Y).` is the logical implication `∀X ∀Y (Connected(X,Y) → CanMove(X,Y))` written "backwards": the `:-` is `←` ("if"). The head is the conclusion, the body is the condition; variables are universally quantified. It is a Horn clause.

## Task 7 – Using Prolog to check a proposed plan
The Python plan's movements are `Move(a,b), Move(b,c)`.

| Query | Expected |
|---|---|
| `?- valid_move(a,b).` | `true.` |
| `?- valid_move(b,c).` | `true.` |
| `?- valid_move(a,c).` | `false.` |
| `?- valid_route([a,b,c]).` | `true.` |
| `?- valid_route([a,c]).` | `false.` |

**Challenge – `Move(a,c)` proposed by the planner:** `valid_move(a,c)` fails, so the warehouse knowledge does **not support** that action; Prolog rejects it. This is *generate → independent verification*: the planner produced a candidate, and a separate logical description of the warehouse checks it.

In [ ]:
import re
# Python cross-check of the expected Prolog answers (this is NOT Prolog; it only mirrors the same facts)
connected_facts = {("a","b"), ("b","a"), ("b","c"), ("c","b")}
valid_move = lambda x, y: (x, y) in connected_facts

for q in [("a","b"), ("b","c"), ("a","c")]:
    print(f"valid_move{q}  ->", valid_move(*q))

# Independent check of the moves in the plan the planner produced
plan_moves = [tuple(m.lower() for m in re.findall(r"[A-C]", str(a))) for a in resA["plan"] if str(a).startswith("Move")]
print("\nMoves in the planner's plan:", plan_moves, "-> all supported:", all(valid_move(*m) for m in plan_moves))

# A hypothetical bad proposal
print("Proposed Move(a,c) supported?", valid_move("a", "c"))

## Task 8 – Connect Prolog to logical reasoning

```prolog
wet_road.
slippery :- wet_road.
reduce_speed :- slippery.
?- reduce_speed.
```
The query succeeds because `reduce_speed` can be derived from the facts and rules. Prolog works backwards: to prove `reduce_speed`, it needs `slippery`; to prove `slippery`, it needs `wet_road`; `wet_road` is a fact, so each goal is discharged in turn.

Reasoning as a chain of implications (forwards):

**wet_road** (fact) **⇒** (wet_road → slippery) **slippery ⇒** (slippery → reduce_speed) **reduce_speed** (conclusion)

i.e. *Fact ⇒ Rule ⇒ Rule ⇒ Conclusion*, which is two steps of *modus ponens*.

## Prolog reflection

**1. Fact vs rule?** A fact is an unconditional statement (`connected(a,b).`); a rule (`head :- body.`) says the head holds *if* the body holds, so it lets new facts be derived from others.

**2. How does a query correspond to "does it follow from the knowledge base"?** `?- q.` asks whether `q` can be derived from the facts and rules (KB ⊢ q). `true` means a derivation was found; `false` means none was found (not necessarily that `q` is false in the real world).

**3. Why verify a Python-generated plan with Prolog?** The verifier works from a different representation and mechanism than the generator, so an error in the planner code (for example a missing precondition check) is unlikely to be repeated in the same way. It also makes the rules explicit and checkable by anyone who reads the facts.

**4. Advantage of an independent verifier when an LLM helped generate the plan?** The plan or code came from a system that can be fluent and wrong. A verifier does not care how convincing the generation looked; it only accepts what follows from the stated knowledge. That gives an answer that does not depend on trusting the LLM. (Its limits: it is only as correct as the facts and rules it is given, and Prolog's negation-as-failure is not the same as classical logic.)

*An AI system can generate a candidate solution, while a separate logical system checks it.*

## Summary: who did what
| | Detail |
|---|---|
| **Written by me** | Problem specification (Task 0), hand plan (Task 1), test plan, answers |
| **Generated by the LLM** | The planner code (`Action`, `bfs_plan`, `is_applicable`, `apply_action`), explanations, the plan checker |
| **Modified by me** | Added the precondition-check switch, `verify_plan`, reachable-state count, assert-based tests |
| **Tested** | Tests A, B1, B2, C1, C2; buggy planner; sheet's example plan; negative preconditions; explanation vs executed states |